# 03 — Stage 2a Severity Classifier Training & Analysis

**YOLOv8n-cls** trained to classify pothole crops as `Low`, `Medium`, or `High` severity.

This model runs **after** the Stage 1 detector crops a pothole region from the frame.

---

## How to use this notebook

| Goal | What to do |
|---|---|
| First training run | Run all cells top to bottom |
| Already trained, view results | Run cells 1–2, then jump to **Section 4** |
| Re-train with different settings | Change values in **Section 2**, re-run Section 3 |
| Improve weak class | Check confusion matrix, look for imbalanced classes |


## Section 1 — Imports & GPU Check

In [ ]:
import sys, time, shutil
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
import torch

BASE_DIR   = Path('..').resolve()
MODELS_DIR = BASE_DIR / 'models'
SEVERITY   = BASE_DIR / 'data/processed/severity_crops'

print('=' * 55)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU  : {p.name}')
    print(f'  VRAM : {p.total_memory/1e9:.1f} GB')
else:
    print('  No GPU')

assert SEVERITY.exists(), (
    'severity_crops/ not found.\nRun: python src/prepare_data.py'
)

# Show crop counts
print('\n  Severity crop counts:')
total = 0
for split in ['train', 'val', 'test']:
    row = []
    for cls in ['Low', 'Medium', 'High']:
        n = len(list((SEVERITY / split / cls).glob('*')))
        total += n
        row.append(f'{cls}:{n}')
    print(f'    {split:5s} → {"  ".join(row)}')
print(f'    Total : {total}')

## Section 2 — ⚙️ Configure Training Parameters

**Change values here before running training.**


In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │          CHANGE THESE VALUES TO TUNE TRAINING                   │
# └─────────────────────────────────────────────────────────────────┘

EPOCHS  = 50     # 50 is usually enough. Increase to 100 if val accuracy is still rising.

BATCH   = 128    # 128 is safe for crops (tiny 128px images, any GPU handles it)
                 # Reduce to 64 only if OOM

IMGSZ   = 128    # Keep at 128 — that's the crop size

# ─────────────────────────────────────────────────────────────────
from train import _gpu_info
_, n_gpus, vram_gb = _gpu_info()

n_train = sum(len(list((SEVERITY/'train'/c).glob('*'))) for c in ['Low','Medium','High'])
steps   = n_train // BATCH
est_min = steps * 50 * 0.005 / 60   # rough: 5ms/batch for small crops

print('  Severity training preview:')
print(f'    Epochs       : {EPOCHS}')
print(f'    Batch        : {BATCH}')
print(f'    Image size   : {IMGSZ}px')
print(f'    Train images : {n_train}')
print(f'    Steps/epoch  : {steps}')
print(f'    Est total    : ~5–10 min')

### Sample Crops — Inspect Before Training

Check that the auto-labelling looks correct. Larger visible potholes should be `High`.

In [ ]:
sev_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
fig, axes  = plt.subplots(3, 6, figsize=(15, 8))

for row, sev in enumerate(['Low', 'Medium', 'High']):
    samples = sorted((SEVERITY / 'train' / sev).glob('*'))[:6]
    for col in range(6):
        ax = axes[row][col]
        if col < len(samples):
            img = cv2.cvtColor(cv2.imread(str(samples[col])), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(sev, color=sev_colors[sev], fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('Severity Crop Samples — verify labelling looks correct', fontsize=12)
plt.tight_layout(); plt.show()
print('\n  If labelling looks wrong, adjust thresholds in src/prepare_data.py:')
print('    SEVERITY_LOW_MAX    = 0.03   (box < 3% of image → Low)')
print('    SEVERITY_MEDIUM_MAX = 0.12   (box < 12% of image → Medium)')

## Section 3 — Train

> **Skip this section if already trained.** Jump to Section 4.


In [ ]:
from train import train_severity

t0 = time.time()
results = train_severity(run_eval=False)
elapsed = time.time() - t0

print(f'\n  Training done in {elapsed/60:.1f} min')

# Copy best weights
best = BASE_DIR / 'runs/severity/weights/best.pt'
dest = MODELS_DIR / 'severity_model.pt'
if best.exists():
    shutil.copy2(best, dest)
    print(f'  Saved → {dest}')

---
## Section 4 — Results & Analysis

Run from here if you already trained.

In [ ]:
# Copy weights if trained outside notebook
best = BASE_DIR / 'runs/severity/weights/best.pt'
dest = MODELS_DIR / 'severity_model.pt'
if best.exists() and not dest.exists():
    shutil.copy2(best, dest)
    print(f'  Copied → {dest}')

results_csv = BASE_DIR / 'runs/severity/results.csv'
if not results_csv.exists():
    print('  No results.csv found. Run Section 3 first.')
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f'  Results loaded: {len(df)} epochs')

    acc_col = 'metrics/accuracy_top1'
    if acc_col in df.columns:
        best_ep  = df[acc_col].idxmax()
        best_acc = df[acc_col].max()
        last_acc = df[acc_col].iloc[-1]
        trend    = df[acc_col].iloc[-5:].diff().mean()
        print(f'\n  Best Top-1 Acc : {best_acc:.4f}  ({best_acc*100:.1f}%)  at epoch {best_ep}')
        print(f'  Last Acc       : {last_acc:.4f}')
        print(f'  Last 5ep trend : {trend:+.4f}  ', end='')
        if trend > 0.002:
            print('↑ Still improving — consider more epochs')
        elif trend > -0.001:
            print('→ Converged')
        else:
            print('↓ Declining')

### 4a — Training Curves

**What to look for:**
- Loss should fall and level off smoothly
- Top-1 accuracy should rise — 70%+ is good for a 3-class problem
- If accuracy is still rising at ep50 → set `EPOCHS = 100` and re-run


In [ ]:
if 'df' not in dir():
    print('Run the cell above first'); raise SystemExit

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [
    ('train/loss',            'val/loss',  'Loss'),
    ('metrics/accuracy_top1', None,        'Top-1 Accuracy'),
    ('metrics/accuracy_top5', None,        'Top-5 Accuracy'),
]
for ax, (tc, vc, title) in zip(axes, pairs):
    if tc in df.columns:
        ax.plot(df['epoch'], df[tc], label='train', color='#3498db', linewidth=1.5)
    if vc and vc in df.columns:
        ax.plot(df['epoch'], df[vc], label='val',   color='#e74c3c',
                linestyle='--', linewidth=1.5)
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

acc_col = 'metrics/accuracy_top1'
if acc_col in df.columns:
    best_ep  = df[acc_col].idxmax()
    best_acc = df[acc_col].max()
    axes[1].axvline(best_ep, color='gold', linestyle=':', linewidth=2,
                    label=f'Best ep {best_ep}  acc={best_acc:.3f}')
    axes[1].legend(fontsize=8)

plt.suptitle('Stage 2a Severity Classifier — Training Curves', fontsize=13)
plt.tight_layout(); plt.show()

### 4b — Confusion Matrix on Test Set

**What to look for:**
- Diagonal should be dark (correct predictions)
- Common mistake: `Low` predicted as `Medium` → this is acceptable (adjacent classes)
- Dangerous mistake: `High` predicted as `Low` → under-detecting severe potholes
- If `High` is weak → that class has fewest samples (~46 test), consider collecting more High-severity data


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

model_path = MODELS_DIR / 'severity_model.pt'
if not model_path.exists():
    print('severity_model.pt not found. Run Section 3 first.')
else:
    model = YOLO(str(model_path))

    CNN_IDX = {0: 'High', 1: 'Low', 2: 'Medium'}   # alphabetical folder order
    CLASSES  = ['Low', 'Medium', 'High']

    test_imgs, y_true = [], []
    for cls in CLASSES:
        for p in sorted((SEVERITY / 'test' / cls).glob('*')):
            if p.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
            img = cv2.imread(str(p))
            if img is not None:
                test_imgs.append(img); y_true.append(cls)

    # Batched inference
    y_pred = []
    for i in range(0, len(test_imgs), 64):
        for r in model(test_imgs[i:i+64], imgsz=128, verbose=False):
            y_pred.append(CNN_IDX[int(r.probs.top1)])

    acc = sum(t==p for t,p in zip(y_true,y_pred)) / len(y_true)
    print(f'  Test accuracy : {acc:.4f} ({acc*100:.1f}%)  on {len(y_true)} crops\n')

    cm   = confusion_matrix(y_true, y_pred, labels=CLASSES)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Severity Confusion Matrix  (accuracy={acc:.2%})')
    plt.tight_layout(); plt.show()

### 4c — Per-Class Report

In [ ]:
if 'y_true' in dir() and 'y_pred' in dir():
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    # Identify weakest class
    from sklearn.metrics import f1_score
    f1s     = f1_score(y_true, y_pred, labels=CLASSES, average=None)
    weakest = CLASSES[f1s.argmin()]
    print(f'\n  Weakest class : {weakest}  (F1={f1s.min():.3f})')
    counts = {c: y_true.count(c) for c in CLASSES}
    print(f'  Sample counts : {counts}')
    if f1s.min() < 0.50:
        print(f'\n  Suggestion: {weakest} has low F1.')
        if counts[weakest] < 100:
            print(f'  → Collect more {weakest} severity pothole images')
        else:
            print(f'  → Try increasing EPOCHS to 100 and re-training')

---
## Section 5 — What to Do Next

| Accuracy result | Recommendation |
|---|---|
| < 55% | Need more epochs or more training data |
| 55–70% | Acceptable, check which class is weak |
| > 70% | Good — proceed to benchmark |

**Next step:** Run `04-benchmark.ipynb` to compare this CNN against the classical CV method.
The better one is used automatically in the pipeline.


In [ ]:
if 'acc' in dir():
    print(f'Severity classifier accuracy: {acc:.2%}')
    print()
    if acc >= 0.70:
        print('  Ready for benchmark!')
        print('  Next: open 04-benchmark.ipynb')
    elif acc >= 0.55:
        print('  Acceptable result. To improve:')
        print('  Change: EPOCHS = 100  (re-run Section 2 + 3)')
    else:
        print('  Needs improvement. Try:')
        print('  1. EPOCHS = 100')
        print('  2. Check confusion matrix — which class is failing?')